In [27]:
from math import factorial

from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window

import ConnectionConfig as cc
cc.setupEnvironment()


Environment variables are set...


In [28]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [29]:
spark = cc.startLocalCluster("FACT_TREASURE_FOUND",4)
spark.getActiveSession()

In [30]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [32]:
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")

In [33]:
dateDimDf.show()

+--------------------+------+---+----+---------+----+--------------+------------+---------+
|          DateSurKey|DateId|Day|Week|    Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|
+--------------------+------+---+----+---------+----+--------------+------------+---------+
|279202f0-8b59-488...|     1| 11|  37|September|2020|             9|           5|     true|
|83cbc09f-40a1-459...|     2| 12|  37|September|2020|             9|           6|    false|
|b4aedc09-0b9c-42e...|     3| 13|  37|September|2020|             9|           7|    false|
|8b04199c-026b-404...|     4| 14|  38|September|2020|             9|           1|     true|
|0763d129-2b1d-47c...|     5| 15|  38|September|2020|             9|           2|     true|
|3ac5d5eb-655e-46d...|     6| 16|  38|September|2020|             9|           3|     true|
|672037d8-2a53-4ba...|     7| 17|  38|September|2020|             9|           4|     true|
|4f3f4e35-4a28-40e...|     8| 18|  38|September|2020|             9|           5

In [34]:
rainDimDf.show()

+----------+--------+--------------------+
|RainSurKey|RainCode|     RainDescription|
+----------+--------+--------------------+
|         1|    RAIN|Weer met regen (c...|
|         2|  NORAIN|   Weer zonder regen|
|         3| UNKNOWN|Regen situatie on...|
+----------+--------+--------------------+



In [35]:
seasonDimDf.show()

+----------+--------------------+
|SeasonName|        SeasonSurKey|
+----------+--------------------+
|     Zomer|ebb03bb0-7282-4d0...|
|    Herfst|617c7216-4bd8-41c...|
|    Winter|b8d3d35c-65b8-438...|
|     Lente|4d9bb07a-f4a2-4f4...|
+----------+--------------------+



In [36]:
fact_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select id, log_type, log_time, treasure_id from treasure_log) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
    .filter(col("log_type") == 2) # log_type 2 is 'Treasure Found'
    .withColumnRenamed("log_time", "LogDate") # Hernoem voor duidelijkheid
    .withColumn("LogDate_date", to_date(col("LogDate"))) # Datumdeel voor joins
)
print("fact_src preview:")
fact_src.show(5)

fact_src preview:
+--------------------+--------+--------------------+--------------------+------------+
|                  id|log_type|             LogDate|         treasure_id|LogDate_date|
+--------------------+--------+--------------------+--------------------+------------+
|[00 65 F4 FF 43 D...|       2|2020-12-11 19:28:...|[A0 3F D6 4E 41 A...|  2020-12-11|
|[00 66 05 9F 29 A...|       2|2021-01-28 22:27:...|[DF 30 9C 0A 82 A...|  2021-01-28|
|[00 66 05 BE 04 2...|       2|2020-10-07 01:41:...|[6C 35 42 80 CC 7...|  2020-10-07|
|[00 66 07 25 01 D...|       2|2020-09-24 06:27:...|[B5 42 46 80 7E 8...|  2020-09-24|
|[00 66 0E 78 83 4...|       2|2022-02-02 19:15:...|[66 8B 62 EC C0 0...|  2022-02-02|
+--------------------+--------+--------------------+--------------------+------------+
only showing top 5 rows


In [37]:

treasure_stages_df = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option("dbtable", "treasure_stages") # Direct de tabelnaam
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("treasure_stages_df preview:")
treasure_stages_df.show(5)
treasure_stages_df.createOrReplaceTempView("treasure_stages")

stage_df = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option("dbtable", "stage") # Direct de tabelnaam
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("stage_df preview:")
stage_df.show(5)
stage_df.createOrReplaceTempView("stage")


treasure_stages_df preview:
+--------------------+--------------------+
|         treasure_id|           stages_id|
+--------------------+--------------------+
|[00 00 3E 2C B1 4...|[62 EA 28 A8 51 3...|
|[00 00 3E 2C B1 4...|[A0 96 DC 61 F0 A...|
|[00 00 3E 2C B1 4...|[AA C9 B1 42 CB 8...|
|[00 00 3E 2C B1 4...|[FD 9F F6 5F EA 6...|
|[00 03 72 3C 7C C...|[1B 32 E8 9A A6 2...|
+--------------------+--------------------+
only showing top 5 rows
stage_df preview:
+--------------------+--------------+--------------------+------------------+------------------+---------------+----+----------+
|                  id|container_size|         description|          latitude|         longitude|sequence_number|type|visibility|
+--------------------+--------------+--------------------+------------------+------------------+---------------+----+----------+
|[00 8F 79 6A C9 2...|             1|In voluptate earu...|  78.2210727266575|15.637835959156016|              3|   0|         2|
|[00 8F 89 FB 21 0

In [38]:
# Vind de laatste stage (hoogste sequence_number) voor elke treasure_id
window_spec_stage = Window.partitionBy("treasure_id").orderBy(col("sequence_number").desc())
last_stage_per_treasure = (
    spark.sql("""
        SELECT
            ts.treasure_id,
            s.id as stage_id,
            s.sequence_number,
            s.latitude,
            s.longitude
        FROM
            treasure_stages ts
        JOIN
            stage s ON ts.stages_id = s.id
    """)
    .withColumn("rn", row_number().over(window_spec_stage))
    .filter(col("rn") == 1)
    .drop("rn", "sequence_number", "stage_id")
    .withColumnRenamed("city", "TreasureCity")
    .withColumnRenamed("country", "TreasureCountry")
)
print("last_stage_per_treasure preview:")
last_stage_per_treasure.show()
last_stage_per_treasure.createOrReplaceTempView("last_stage_per_treasure")

last_stage_per_treasure preview:
+--------------------+------------------+-------------------+
|         treasure_id|          latitude|          longitude|
+--------------------+------------------+-------------------+
|[00 03 72 3C 7C C...|12.568262445880535|  77.92589024707661|
|[00 04 62 1B 29 E...|51.728693518723624|-2.1977132499510175|
|[00 06 05 10 FC 0...| 34.06963357435847| -89.88196811630556|
|[00 08 A9 BF BE 4...| 22.77425290662822|  88.19301469524981|
|[00 08 B9 20 1B F...|27.994826260609962|  -82.2164858018488|
|[00 09 57 54 16 D...|11.168920300382176|  77.60795611973033|
|[00 0B 12 E3 93 2...|30.235295899656442|  78.14218071988124|
|[00 0E 3E 40 56 D...| 34.93535875366291|  -79.7625650335257|
|[00 0F 1D 4F FF A...|-3.803613250238623| -45.23241860685075|
|[00 0F 44 E2 4E D...|10.844275178123029|  78.77381030667783|
|[00 10 05 86 4E 3...| 34.16628716054675| -95.37296713235307|
|[00 11 6E F0 A5 0...| 63.13177486192953|  29.99343275870074|
|[00 11 E5 15 88 8...| 41.03316966236

In [39]:
# --- Start Fact Table constructie ---
# De basis van onze facttabel, met de originele log data en treasure coördinaten
fact_enriched = fact_src.join(
    last_stage_per_treasure,
    on="treasure_id",
    how="left"
).drop("log_type", "treasure_id") # Drop kolommen die niet in de fact horen


In [42]:


# 1️⃣ Eerst extraheer dag, week en jaar uit LogDate_date (tijd negeren)
fact_enriched_clean = (
    fact_enriched
    .withColumn("log_day", dayofmonth("LogDate_date"))
    .withColumn("log_week", weekofyear("LogDate_date"))
    .withColumn("log_year", year("LogDate_date"))
)

# 2️⃣ Join met DateDim op dag/week/jaar
fact_with_date_key = (
    fact_enriched_clean.join(
        dateDimDf.select("DateSurKey", "Day", "Week", "Year", "MonthOfTheYear"),
        (fact_enriched_clean["log_day"] == dateDimDf["Day"]) &
        (fact_enriched_clean["log_week"] == dateDimDf["Week"]) &
        (fact_enriched_clean["log_year"] == dateDimDf["Year"]),
        how="left"
    )
    .drop("log_day", "log_week", "log_year", "Day", "Week", "Year")  # opruimen
)

print("fact_with_date_key preview:")
fact_with_date_key.show()


fact_with_date_key preview:
+--------------------+--------------------+------------+------------------+-------------------+--------------------+--------------+
|                  id|             LogDate|LogDate_date|          latitude|          longitude|          DateSurKey|MonthOfTheYear|
+--------------------+--------------------+------------+------------------+-------------------+--------------------+--------------+
|[00 66 57 F2 E5 7...|2023-03-12 05:22:...|  2023-03-12|40.226554831806695| 38.861962880368225|ef836c82-8452-416...|             3|
|[00 66 35 0A FB 3...|2022-03-17 01:50:...|  2022-03-17|24.612091799543215|  93.88416601962795|61548f59-e626-4b8...|             3|
|[00 66 60 98 19 7...|2020-10-24 22:13:...|  2020-10-24|24.858341243384775| -99.56011574602604|60bed575-7abe-43b...|            10|
|[00 66 05 BE 04 2...|2020-10-07 01:41:...|  2020-10-07| 33.83879413280263| -96.36389526445393|1c82f1eb-6db3-4f4...|            10|
|[00 66 2F A5 BA B...|2022-02-15 19:35:...|  202